In [ ]:
import sys
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/pvc-6/pvc6_helper_modules/')



from pvc6_stim_analysis import *
from pvc6_load_data import *
from pvc6_plotting import *


%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from spikeparam.patch.fit import Spike, SpikeGroup
from tqdm.notebook import tqdm
import pickle

import seaborn as sns
sns.set(rc={'figure.figsize':(12,9)})
sns.set_style('white')

import warnings
warnings.filterwarnings('ignore')
import os

set_plot_style()


In [ ]:
# ── Run-control flags ─────────────────────────────────────────────────────────
FORCE_RERUN = False   # set True to recompute and overwrite pickles
PICKLE_DIR  = "/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles/"


## 1. Data Source: PVC-6 (CRCNS) — Cell 2

Same dataset and analysis pipeline as cell 1, applied to a second recording. Replicating across cells tests whether stimulus-driven waveform variability is a general property of cortical neurons.

## 2. Data Loading & QC

### 2a. Raw Data

We open the HDF5 recording and extract sweep-level ephys + stimulus traces at 200 kHz.

In [ ]:
fs = 200000 # sampling rate
one_ms = int(fs / 1000)
total_sweeps = 66
filename =  '/Users/blancamartin/Downloads/151145.04_Data.h5' # filename
f = h5py.File(filename, 'r')

## 3. Stimulus Labeling & Spike Extraction

For each sweep we label the stimulus type (constant / ramp / pink) and extract pink noise spectral properties per spike. Note: three sweeps (27, 47, 50) are removed due to incorrect spike counts in the raw data.

In [ ]:
# Process sweeps and extract features
pink_types = process_pink_type_info(f, fs)

### 3b. Processing Sweeps & Loading Pickles

We extract every spike from every sweep and save to pickles for fast reloading.

In [ ]:
_pickle_paths = {
    'stim': os.path.join(PICKLE_DIR, 'df_stim_features2.pkl'),
    'data': os.path.join(PICKLE_DIR, 'all_data2.pkl'),
}
df_stim_features, all_data_flat = load_or_compute_cell_data(
    _pickle_paths, FORCE_RERUN,
    n_sweeps=66, f=f, one_ms=one_ms, fs=fs, pink_types=pink_types,
)

pickle_path_waveform = os.path.join(PICKLE_DIR, 'stim_waveforms2.pkl')
if not FORCE_RERUN and os.path.exists(pickle_path_waveform):
    try:
        with open(pickle_path_waveform, 'rb') as file:
            stim_waveforms = pickle.load(file)
        all_contant_spks, all_ramp_spks, all_pink_spks = stim_waveforms
        print('Loaded stim_waveforms from pickle.')
    except Exception as _e:
        print(f'stim_waveforms pickle load failed ({_e}); recomputing from all_data.')
        stim_waveforms = None
else:
    stim_waveforms = None

if stim_waveforms is None:
    pink_types_local = process_pink_type_info(f, fs)
    (_, _, _, _, _, _, _, _, _, _, _, _, _,
     _, _, _, all_contant_spks, all_ramp_spks, all_pink_spks
    ) = process_sweeps(66, f, one_ms, fs, pink_types_local, load_sweep, find_spike_times, fit_exp_nonlinear)
    stim_waveforms = [all_contant_spks, all_ramp_spks, all_pink_spks]
    with open(pickle_path_waveform, 'wb') as file:
        pickle.dump(stim_waveforms, file)
    print('Saved stim_waveforms to pickle.')


### 3c. Data Cleaning

Sweeps 27, 47, and 50 have incorrect spike counts. We drop these rows before fitting.

## 4. Spike Waveform Parameterization

### 4b. Fitting Spike Waveforms

We fit every spike with the `Spike` class, extracting peak amplitude, sharpness, width, exp_lambda, exp_const, inflection time, and log ISI.

In [ ]:
sp = Spike(thresh_amp= -10, window_length=(5., 5.), smooth_frac=.008, pre_inflection_ms = 0.5)
sp.fit(all_data_flat, fs, n_jobs=-1, progress=tqdm)

In [ ]:
sp.plot(show_points=True)

In [ ]:
sp.filter_features()

In [ ]:
df_features = sp.df_features

### 4b-i. Fit Quality Distributions

R² histograms for the ramp-amplitude (linear) and exp-decay fits, and distributions of each extracted spike waveform feature.

In [ ]:
plot_fit_quality_distributions(sp)
plot_feature_histograms(df_features)


### 4c. Merging Spike Features with Stimulus Metadata

In [ ]:
df = pd.concat([df_stim_features, df_features], axis=1)
df = df.dropna(subset=['stim_type'])

In [ ]:
# Assuming your DataFrame is called df and the column for sweeps is 'sweep'

# Identify rows that belong to the sweeps where one row needs to be removed
sweeps_to_fix = [50, 47, 27]

# Create a mask that finds the first occurrence of each sweep in sweeps_to_fix
mask = df_stim_features[df_stim_features['sweep'].isin(sweeps_to_fix)].groupby('sweep').head(1).index

# Drop the first occurrence for each incorrect sweep
df_stim_features = df_stim_features.drop(mask)

# Reset index after removal (optional)
df_stim_features = df_stim_features.reset_index(drop=True)



### 5b. Pink Noise Spikes — Waveform Gallery

In [ ]:
pink_spikes = df[df['stim_type'] == 'pink']

# Get the indices to plot
indices_to_plot = pink_spikes.index.tolist()

### 5c. Excluding Last-Spike-of-Sweep Artifacts

In [ ]:
last_spikes_indices = df.groupby('sweep').tail(1).index
print("Indices of last spikes in each sweep:", last_spikes_indices)

### 5e. Correlation Scatter Plots — Pink Spikes (Excluding Last-Sweep Spikes)

In [ ]:
df_pink = df.loc[df['stim_type'] == 'pink']
# Find which indices are actually present in df_pink, remove last spike of sweeps
valid_indices = [idx for idx in last_spikes_indices if idx in df_pink.index]

# Drop the last spikes
df_pink_filtered = df_pink.drop(valid_indices)

df_pink_first = df_pink.loc[df_pink['spike_num'] == 0].copy()

In [ ]:
plot_top_correlations_by_window(df_pink_filtered, window_ms=5)


In [ ]:
#get dfs for regression models 

df_pink_no_stim_filtered = df_pink_filtered.drop(['stim_exp', 'stim_mean', 'stim_std' ], axis=1)
df_pink_no_stim_first = df_pink_first.drop(['stim_exp', 'stim_mean', 'stim_std' ], axis=1)
df_pink_onlystim_filtered = df_pink_filtered.drop(['ramp_amp', 'inflection_time', 'inflection_amp', 'peak_amp', 'peak_width','peak_sharpness', 'exp_lambda', 'exp_const' ], axis=1)
df_pink_onlystim_first = df_pink_first.drop(['ramp_amp', 'inflection_time', 'inflection_amp', 'peak_amp', 'peak_width','peak_sharpness', 'exp_lambda', 'exp_const' ], axis=1)


## Supplementary Figures — Cell 2

### 2a. Constant Current — Baseline Waveform

In [ ]:
sweep_number = 54
index_start = 403900
index_end = 406000
dset, times = load_sweep(sweep_number, f, fs)
plot_ephys_data(dset, times, index_start, index_end)
plot_stim_data(dset, times, index_start, index_end)
plt.show()

### 2b. Pink Noise — Stimulus-Driven Variability

In [ ]:
sweep_number = 65
index_start = 0
index_end = int(10e10)
dset, times = load_sweep(sweep_number, f, fs)
plot_ephys_data(dset, times, index_start, index_end)
plot_stim_data(dset, times, index_start, index_end)
plt.show()

In [ ]:
all_contant_spks, all_ramp_spks, all_pink_spks = stim_waveforms[0], stim_waveforms[1], stim_waveforms[2]

In [ ]:
plot_spike_and_derivative(28, f, fs, one_ms)

## 5. Exploratory Analysis

### 5a. Average Waveform by Stimulus Type

In [ ]:
plot_avg_waveform_by_stim_type(all_contant_spks, all_ramp_spks,all_pink_spks)

### 5b. Feature Correlation Structure

#### i. All Spikes (constant + ramp + pink combined)

In [ ]:
df_all = df.drop(['sweep', 'stim_type', 'pink_type', 'spike_num'], axis=1)
axes = pd.plotting.scatter_matrix(df_all, alpha=0.5, figsize=(12, 12))
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(45)
    ax.xaxis.label.set_ha('right')
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')
plt.tight_layout()
plt.show()
spikes_corr = df_all.corr()
sns.heatmap(spikes_corr, xticklabels=spikes_corr.columns, yticklabels=spikes_corr.columns, annot=True)
plt.show()
rho = df_all.corr()
pval = df_all.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*rho.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [0.001,0.01,0.05] if x<=t]))
rho.round(2).astype(str) + p

#### ii. Pink Noise Spikes Only

In [ ]:
df_pink = df.loc[df['stim_type'] == 'pink']
df_pink = df_pink.drop(['sweep', 'stim_type', 'pink_type', 'spike_num'], axis=1)
axes = pd.plotting.scatter_matrix(df_pink, alpha=0.5, figsize=(12, 12))
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(45)
    ax.xaxis.label.set_ha('right')
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')
plt.tight_layout()
plt.show()
spikes_corr = df_pink.corr()
sns.heatmap(spikes_corr, xticklabels=spikes_corr.columns, yticklabels=spikes_corr.columns, annot=True)
plt.show()
rho = df_pink.corr()
pval = df_pink.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*rho.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [0.001,0.01,0.05] if x<=t]))
rho.round(2).astype(str) + p

#### iii. First Spike per Pink Sweep — Minimizing Adaptation

In [ ]:
df_pink_first = df.loc[df['stim_type'] == 'pink']
df_pink_first = df_pink_first.loc[df['spike_num'] == 0]
df_pink_first = df_pink_first.drop(['sweep', 'stim_type', 'spike_num', 'pink_type',], axis=1)
axes = pd.plotting.scatter_matrix(df_pink_first, alpha=0.5, figsize=(12, 12))
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(45)
    ax.xaxis.label.set_ha('right')
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')
plt.tight_layout()
plt.show()
spikes_corr = df_pink_first.corr()
sns.heatmap(spikes_corr, xticklabels=spikes_corr.columns, yticklabels=spikes_corr.columns, annot=True)
plt.show()
rho = df_pink_first.corr()
pval = df_pink_first.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*rho.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [0.001,0.01,0.05] if x<=t]))
rho.round(2).astype(str) + p

In [ ]:
# Calculate the number of spikes in each sweep
spike_counts = df.groupby('sweep')['spike_num'].count()

# Calculate the duration of each sweep in seconds using the indices from spike_counts
sweep_durations = {sweep: f[f"Sweep_{int(sweep)}"].shape[0] / fs for sweep in spike_counts.index}

# Calculate the firing rate for each sweep
firing_rates = {sweep: spike_counts[sweep] / sweep_durations[sweep] for sweep in spike_counts.index}

# Convert the dictionary to a DataFrame for easier handling and visualization
firing_rate_df = pd.DataFrame(list(firing_rates.items()), columns=['Sweep', 'Firing Rate'])

# Extract 'stim_type' values for each unique 'sweep'
stim_type_series = df.groupby('sweep')['stim_type'].first()
# Add the 'stim_type' to the firing_rate_df
firing_rate_df['Stim Type'] = firing_rate_df['Sweep'].map(stim_type_series)

# Display the updated DataFrame
print(firing_rate_df)

## 8. Regression: Predicting Log ISI

Log ISI is a proxy for instantaneous firing rate. We fit three ridge regression models: (1) spike waveform features only, (2) stimulus features only, (3) both combined.

### 8a. Model 1 — Spike Waveform Features Only

In [ ]:
y_log_isi = df_pink_filtered['log_isi']

pickle_path = os.path.join(PICKLE_DIR, 'ridge_results_spike_only2.pkl')
if not FORCE_RERUN and os.path.exists(pickle_path):
    with open(pickle_path, 'rb') as file:
        ridge_results_spike_only = pickle.load(file)
    print('Loaded ridge_results_spike_only from pickle.')
else:
    X_spike_only = df_pink_filtered.drop(columns=['log_isi', 'stim_exp', 'stim_mean', 'stim_std'])
    ridge_results_spike_only = run_ridge_regression_kfold(X_spike_only, y_log_isi)
    with open(pickle_path, 'wb') as file:
        pickle.dump(ridge_results_spike_only, file)
    print('Saved ridge_results_spike_only to pickle.')


In [ ]:
plot_ridge_results_grid(
    {'M1_spike_only': ridge_results_spike_only,
     'M2_spike_stim': ridge_results_spike_stim,
     'M3_stim_only':  ridge_results_stim_only},
    keys=['M1_spike_only', 'M2_spike_stim', 'M3_stim_only'],
    labels=['Spike only', 'Spike + stim', 'Stim only'],
    ys={'M1_spike_only': y_log_isi,
        'M2_spike_stim': y_log_isi,
        'M3_stim_only':  y_log_isi},
)


### 8b. Model 2 — Spike + Stimulus Features

In [ ]:
pickle_path = os.path.join(PICKLE_DIR, 'ridge_results_spike_stim2.pkl')
if not FORCE_RERUN and os.path.exists(pickle_path):
    with open(pickle_path, 'rb') as file:
        ridge_results_spike_stim = pickle.load(file)
    print('Loaded ridge_results_spike_stim from pickle.')
else:
    X_spike_stim = df_pink_filtered.drop(columns=['log_isi'])
    ridge_results_spike_stim = run_ridge_regression_kfold(X_spike_stim, y_log_isi)
    with open(pickle_path, 'wb') as file:
        pickle.dump(ridge_results_spike_stim, file)
    print('Saved ridge_results_spike_stim to pickle.')


In [ ]:
# See combined grid above.


### 8c. Model 3 — Stimulus Features Only

In [ ]:
pickle_path = os.path.join(PICKLE_DIR, 'ridge_results_stim_only2.pkl')
if not FORCE_RERUN and os.path.exists(pickle_path):
    with open(pickle_path, 'rb') as file:
        ridge_results_stim_only = pickle.load(file)
    print('Loaded ridge_results_stim_only from pickle.')
else:
    X_stim_only = df_pink_filtered[['stim_exp', 'stim_mean', 'stim_std']]
    ridge_results_stim_only = run_ridge_regression_kfold(X_stim_only, y_log_isi)
    with open(pickle_path, 'wb') as file:
        pickle.dump(ridge_results_stim_only, file)
    print('Saved ridge_results_stim_only to pickle.')


In [ ]:
# FDR correction across ISI models (M1 / M2 / M3)
apply_fdr_pvc6({'M1_spike_only': ridge_results_spike_only,
                'M2_spike_stim': ridge_results_spike_stim,
                'M3_stim_only':  ridge_results_stim_only})

### 8d. Model Comparison

In [ ]:
#Set up dataframes for plotting 

feature_importance_spike_only_df = pd.DataFrame({'Feature': ridge_results_spike_only['feature_names'], 'Coefficient': ridge_results_spike_only['coefficients']})
feature_importance_spike_stim_df = pd.DataFrame({'Feature': ridge_results_spike_stim['feature_names'], 'Coefficient': ridge_results_spike_stim['coefficients']})
feature_importance_stim_only_df = pd.DataFrame({'Feature': ridge_results_stim_only['feature_names'], 'Coefficient': ridge_results_stim_only['coefficients']})

feature_importance_spike_only_df['Model'] = 'No stim features'
feature_importance_spike_stim_df['Model'] = 'With stim features'
feature_importance_stim_only_df['Model'] = 'Only stim features'

In [ ]:
#Plot comparisons
plot_r2_comparison(ridge_results_spike_only, ridge_results_spike_stim, ridge_results_stim_only)
plot_combined_actual_vs_predicted(y_log_isi, ridge_results_spike_only['y_pred_cv'], ridge_results_spike_stim['y_pred_cv'], ridge_results_stim_only['y_pred_cv'])
plot_combined_feature_importance(feature_importance_spike_only_df, feature_importance_spike_stim_df, feature_importance_stim_only_df)
